
**IT Support Ticket Triage & Auto Response Bot**


In [ ]:
!pip install openai anthropic pydantic python-dotenv

In [ ]:
import os

In [ ]:
PROVIDER = "openai"

try:
  from google.colab import userdata

  for name in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
    try:
      os.environ[name] = userdata.get(name)
    except Exception:
      pass
except ImportError:
  from dotenv import load_dotenv
  load_dotenv()


if PROVIDER == "openai" and not os.getenv("OPENAI_API_KEY"):
  print("OPENAI_API_KEY not set")

if PROVIDER == "anthropic" and not os.getenv("ANTHROPIC_API_KEY"):
  print("Anthrophic key is not set")


print("Provider:", PROVIDER)

Provider: openai


Ticket dataset - golden data set

In [ ]:
TICKETS = [
  {"id": 1,  "cat": "Access",   "pri": "P2", "text": "Forgot my password, locked out of Outlook after 3 attempts. Can't reply to clients."},
  {"id": 2,  "cat": "Network",  "pri": "P3", "text": "VPN disconnects every 10 minutes when working from home. Reconnecting works for a while."},
  {"id": 3,  "cat": "Network",  "pri": "P1", "text": "Entire Pune office has no internet since 9:40. Around 200 people idle."},
  {"id": 4,  "cat": "Software", "pri": "P3", "text": "Please install Adobe Acrobat Pro on my laptop for contract reviews starting next week."},
  {"id": 5,  "cat": "Hardware", "pri": "P1", "text": "Laptop battery looks swollen and the bottom is very hot. Screen flickers."},
  {"id": 6,  "cat": "Access",   "pri": "P3", "text": "Grant Finance SharePoint access for new joiner Priya S, starting Monday."},
  {"id": 7,  "cat": "Software", "pri": "P1", "text": "SAP won't launch since last night's update. Billing team (15) can't raise invoices, month-end close today."},
  {"id": 8,  "cat": "Hardware", "pri": "P3", "text": "3rd floor printer keeps jamming. We're using the 2nd floor one for now."},
  {"id": 9,  "cat": "Network",  "pri": "P3", "text": "Can't connect to Wi-Fi in meeting room B. LAN cable works fine."},
  {"id": 10, "cat": "Access",   "pri": "P2", "text": "Got a new phone, now login says 'MFA device not registered'. Can't access anything."},
  {"id": 11, "cat": "Software", "pri": "P2", "text": "Excel crashes every time I open the macro workbook. Report due tomorrow morning."},
  {"id": 12, "cat": "Hardware", "pri": "P3", "text": "URGENT!!!!! MOUSE NOT WORKING!!!"},
  {"id": 13, "cat": "Software", "pri": "P2", "text": "Sir Outlook mein mail nahi aa raha since morning, pura team ka same issue hai."},
  {"id": 14, "cat": "Access",   "pri": "P3", "text": "Need VPN access for the new client project, and also my keyboard's E key is stuck."},
]
print(len(TICKETS), "tickets loaded")

14 tickets loaded


Triage schema

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal, Optional


class TicketTriage(BaseModel):
  category: Literal["Access", "Network", "Software", "Hardware"]
  priority: Literal["P1", "P2", "P3"]
  summary: str = Field(description="One-line restatement of the issue", max_length=120)
  requester_name: Optional[str] = None
  affected_system: Optional[str] = None
  users_affected: Literal['single', 'team', 'org_wide']
  blocking_work: bool
  confidence: float = Field(ge=0, le=1, description="Confidence in the triage")

In [ ]:
try:
  TicketTriage.model_validate_json('{"category": "VPN", "priority": "P3", "summary": "VPN disconnects every 10 minutes when working from home. Reconnecting works for a while."}')
except ValidationError as e:
  print(e.json())

[{"type":"literal_error","loc":["category"],"msg":"Input should be 'Access', 'Network', 'Software' or 'Hardware'","input":"VPN","ctx":{"expected":"'Access', 'Network', 'Software' or 'Hardware'"},"url":"https://errors.pydantic.dev/2.13/v/literal_error"},{"type":"missing","loc":["users_affected"],"msg":"Field required","input":{"category":"VPN","priority":"P3","summary":"VPN disconnects every 10 minutes when working from home. Reconnecting works for a while."},"url":"https://errors.pydantic.dev/2.13/v/missing"},{"type":"missing","loc":["blocking_work"],"msg":"Field required","input":{"category":"VPN","priority":"P3","summary":"VPN disconnects every 10 minutes when working from home. Reconnecting works for a while."},"url":"https://errors.pydantic.dev/2.13/v/missing"},{"type":"missing","loc":["confidence"],"msg":"Field required","input":{"category":"VPN","priority":"P3","summary":"VPN disconnects every 10 minutes when working from home. Reconnecting works for a while."},"url":"https://err

System prompt

In [ ]:
import json

TRIAGE_SYSTEM = f"""You are an L1 IT service desk triage assistant.
Classify each ticket and return ONLY a JSON object matching this schema:
{json.dumps(TicketTriage.model_json_schema(), indent=2)}

Priority rubric:
- P1: whole team or organisation blocked with no workaround, OR any safety/security risk
  (overheating hardware, phishing, data exposure), OR a business deadline today for a team.
- P2: a single user fully blocked, or a team degraded but partly working.
- P3: workaround exists, inconvenience, or a request for future access/installs.
Capital letters, exclamation marks and the word "urgent" do not raise priority on their own.

Categories: Access (passwords, MFA, permissions), Network (internet, VPN, Wi-Fi),
Hardware (physical devices), Software (applications, email clients, updates).
If a ticket has several issues, classify the most urgent one.
Tickets may mix English with Hindi or Kannada. Do not invent names or systems not in the text."""
print(TRIAGE_SYSTEM, "...")

You are an L1 IT service desk triage assistant.
Classify each ticket and return ONLY a JSON object matching this schema:
{
  "properties": {
    "category": {
      "enum": [
        "Access",
        "Network",
        "Software",
        "Hardware"
      ],
      "title": "Category",
      "type": "string"
    },
    "priority": {
      "enum": [
        "P1",
        "P2",
        "P3"
      ],
      "title": "Priority",
      "type": "string"
    },
    "summary": {
      "description": "One-line restatement of the issue",
      "maxLength": 120,
      "title": "Summary",
      "type": "string"
    },
    "requester_name": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Requester Name"
    },
    "affected_system": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": 

One call function

In [ ]:

import re
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
CLAUDE_MODEL = os.getenv("CLAUDE_MODEL", "anthropic/claude-haiku-4-5-20251001")


def _mock_llm(system: str, user: str) -> str:
  """ keyword rules, exists so class can continue without keys """
  t = user.lower()

  if "ResponseDraft" in system or "first response" in system:
    return json.dump({
        "subject": "We've received your ticket",
        "body": "Thanks for reaching out. We've logged your issue and an engineer will follow up.",
        "next_steps": ["Keep this ticket number for reference"],
        "escalate_to_l2": False
    })

  cat = (
      "Access" if re.search(r"password|mfa|access|login|locked", t) else
      "Network" if re.search(r"vpn|wi-?fi|internet|lan", t) else
      "Hardware" if re.search(r"laptop|battery|screen|mouse|keyboard|printer", t) else
      "Software"
  )

  pri = (
      "P1" if re.search(r"entire|whole|office|team \(|swollen|hot)", t) else
      "P2" if re.search(r"can't|cannot|crash|locked", t) else "P3"

  )

  return json.dumps({
      "category": cat,
      "priority": pri,
      "summary": user.split("---")[1].strip()[:110],
      "users_affected": "single",
      "blocking_work": pri != "P3",
      "confidence": 0.5,
  })

def call_llm(system: str, user: str, temperature: float = 0.0) -> str:
  if PROVIDER == "openai":
    from openai import OpenAI

    resp = OpenAI().chat.completions.create(
      model=OPENAI_MODEL,
      response_format={ "type": "json_object" },
      messages=[
        {"role": "system", "content": system},
        {"role": "user", "content": user},
      ],
      temperature=temperature,
    )
    print("openAI response: ", resp)
    return resp.choices[0].message.content

  if PROVIDER == "anthropic":
    import anthropic

    resp = anthropic.Anthropic().completions.create(
      model=CLAUDE_MODEL,
      extra_body={"temperature": temperature},
      max_token=1024,
      system=system,
      messages=[{"content": user, "role": "user"}],
    )
    print("Anthropic response:", resp)
    return resp.content[0].text

  return _mock_llm(system, user)


def extract_json(text: str) -> str:
  """ Strip JSON fences and any chatter around the object """
  start, end = text.find("{"), text.rfind("}")
  return text[start:end+1] if start != -1 and end != -1 else text

raw = call_llm(TRIAGE_SYSTEM, f"Ticket: {TICKETS[6]['text']}")
print(raw)


openAI response:  ChatCompletion(id='chatcmpl-ETV27JFpaUoITaxWfMX84qG7kU46j', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "category": "Software",\n  "priority": "P1",\n  "summary": "SAP won\'t launch after update, blocking billing team from invoicing.",\n  "requester_name": null,\n  "affected_system": "SAP",\n  "users_affected": "team",\n  "blocking_work": true,\n  "confidence": 1\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1790699511, model='gpt-4o-mini-2024-07-18', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_2a9610bf9f', usage=CompletionUsage(completion_tokens=74, prompt_tokens=636, total_tokens=710, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_wr

In [ ]:
first = TicketTriage.model_validate_json(extract_json(raw))
print(first)

category='Software' priority='P1' summary="SAP won't launch after update, blocking billing team from invoicing." requester_name=None affected_system='SAP' users_affected='team' blocking_work=True confidence=1.0


Validate and retry with the error as feedback

In [ ]:
def triage(ticket: str, temperature: float = 0.0, max_retries: int = 2) -> TicketTriage:
  base = f"Ticket: {ticket}"
  error = None
  for attempt in range(max_retries):
    user = base if error is None else (f"{base} -- Your previous JSON failed validation: {error}. Return a corrected JSON object only.")
    raw = call_llm(TRIAGE_SYSTEM, user, temperature)
    try:
      return TicketTriage.model_validate_json(extract_json(raw))
    except ValidationError as e:
      error = str(e)
      print(f"attempt {attempt + 1} failed, retrying...")
  raise RuntimeError(f"Triage failed after {max_retries}")


print(triage(TICKETS[6]["text"], temperature=0.5).model_dump_json(indent=2))




openAI response:  ChatCompletion(id='chatcmpl-ETV2921GNIK1lMtwtD6H6qfPMtO38', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "category": "Software",\n  "priority": "P1",\n  "summary": "SAP won\'t launch after update, blocking billing team from invoicing.",\n  "requester_name": null,\n  "affected_system": "SAP",\n  "users_affected": "team",\n  "blocking_work": true,\n  "confidence": 1\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1790699513, model='gpt-4o-mini-2024-07-18', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_983b0fc8fd', usage=CompletionUsage(completion_tokens=74, prompt_tokens=636, total_tokens=710, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_wr

The auto response draft

In [ ]:
class ResponseDraft(BaseModel):
  subject: str
  body: str = Field(description="One-line restatement of the issue", max_length=120)
  next_steps: list[str] = Field(max_length=4)
  escalate_to_l2: bool


DRAFT_SYSTEM = f"""You write first responses for an IT service desk.
Acknowledge the issue in plain language, give safe self-help steps only if they exist,
and never promise a resolution time. For P1 tickets, say the issue has been escalated.
Return ONLY a JSON object matching:
{json.dumps(ResponseDraft.model_json_schema(), indent=2)}"""


def draft_reply(ticket: str, triage: TicketTriage) -> ResponseDraft:
  user = f"Ticket: {ticket} \n Triage: {triage.model_dump_json(indent=2)}"
  raw = call_llm(DRAFT_SYSTEM, user, temperature=0.4)
  return ResponseDraft.model_validate_json(extract_json(raw))

ticket1 = TICKETS[0]["text"] # Select ticket

triage1 = triage(ticket1) # Get triage

print(triage1.model_dump_json(indent=2))

draft1 = draft_reply(ticket1, triage1) # create the draft response

print(draft1.model_dump_json(indent=2))

openAI response:  ChatCompletion(id='chatcmpl-ETVCPfDxyR3vjkHOEg29SYNt7A3l6', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "category": "Access",\n  "priority": "P2",\n  "summary": "Locked out of Outlook due to forgotten password",\n  "requester_name": null,\n  "affected_system": "Outlook",\n  "users_affected": "single",\n  "blocking_work": true,\n  "confidence": 1\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1790700149, model='gpt-4o-mini-2024-07-18', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_983b0fc8fd', usage=CompletionUsage(completion_tokens=71, prompt_tokens=632, total_tokens=703, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, c

Temperature lab - run 5 times

In [ ]:
from collections import Counter

def consistency(ticket: str, temperature: float, n: int = 5) -> tuple[float, str]:
    labels = [f"{r.category}/{r.priority}"
              for r in (triage(ticket, temperature) for _ in range(n))]
    top_label, count = Counter(labels).most_common(1)[0]
    return count / n, top_label

for idx in (2, 9, 12):              # tickets 3, 10, 13
    for temp in (0.0, 1.0):
        score, label = consistency(TICKETS[idx]["text"], temp)
        print(f"ticket {idx + 1:>2}  temp={temp}  agreement={score:.0%}  top={label}")

openAI response:  ChatCompletion(id='chatcmpl-ETVGSio4Hm939iPJr7h2nKqYygOeU', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "category": "Network",\n  "priority": "P1",\n  "summary": "Entire Pune office has no internet since 9:40.",\n  "requester_name": null,\n  "affected_system": null,\n  "users_affected": "org_wide",\n  "blocking_work": true,\n  "confidence": 1\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1790700400, model='gpt-4o-mini-2024-07-18', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_2a9610bf9f', usage=CompletionUsage(completion_tokens=74, prompt_tokens=632, total_tokens=706, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cach

Batch evaluation and volume view

In [ ]:
import pandas as pd


rows = []

for t in TICKETS:
  res = triage(t["text"])
  rows.append({
      "id": t["id"],
      "gold_cat": t["cat"],
      "pred_cat": res.category,
      "gold_pri": t["pri"],
      "pred_pri": res.priority,
      "conf": res.confidence
  })

df = pd.DataFrame(rows)
print("Category accuracy: ", round((df.gold_cat == df.pred_cat).mean(), 2))
print("Priority accuracy: ", round((df.gold_pri == df.pred_pri).mean(), 2))
print("Confidence accuracy: ", round(df.conf.mean(), 2))

display(df[(df.gold_cat != df.pred_cat) | (df.gold_pri != df.pred_pri)])

openAI response:  ChatCompletion(id='chatcmpl-ETVUn57WzKzsLqfyIVtrIBrRzB7Cw', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='{\n  "category": "Access",\n  "priority": "P2",\n  "summary": "Locked out of Outlook due to forgotten password.",\n  "requester_name": null,\n  "affected_system": "Outlook",\n  "users_affected": "single",\n  "blocking_work": true,\n  "confidence": 1\n}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1790701289, model='gpt-4o-mini-2024-07-18', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_2a9610bf9f', usage=CompletionUsage(completion_tokens=71, prompt_tokens=632, total_tokens=703, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, 

,id,gold_cat,pred_cat,gold_pri,pred_pri,conf
1,2,Network,Network,P3,P2,0.9
8,9,Network,Network,P3,P2,1.0
11,12,Hardware,Hardware,P3,P2,1.0
12,13,Software,Software,P2,P1,1.0


In [ ]:
display(pd.crosstab(df.pred_cat, df.pred_pri, margins=True))

pred_pri,P1,P2,P3,All
pred_cat,,,,
Access,0,2,2,4
Hardware,1,1,1,3
Network,1,2,0,3
Software,2,1,1,4
All,4,6,4,14
